# Оптимизация параметров SIR модели

**Цель:** Поиск оптимальных параметров модели (β_und, detection_time, death_rate)
для минимизации пиковой заболеваемости и смертности с помощью многокритериальной
оптимизации (Pareto).

## Инициализация проекта и загрузка пакетов

In [1]:
using DrWatson
@quickactivate "project"

using BlackBoxOptim, Random, Statistics

include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

## Целевая функция: минимизируем пиковую заболеваемость и смертность

Принимает вектор параметров `x`:
- `x[1]` = β_und (коэффициент заразности невыявленных)
- `x[2]` = detection_time (время до выявления, дней)
- `x[3]` = death_rate (вероятность смерти)

Возвращает кортеж из двух минимизируемых показателей:
1. Пиковая доля инфицированных
2. Доля умерших

In [2]:
function cost_multi(x)
    model = initialize_sir(;
        Ns = [1000, 1000, 1000],
        β_und = fill(x[1], 3),
        β_det = fill(x[1]/10, 3),
        infection_period = 14,
        detection_time = round(Int, x[2]),
        death_rate = x[3],
        reinfection_probability = 0.1,
        Is = [0, 0, 1],
        seed = 42,
        n_steps = 100,
    )

    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    dead_count(model) = 3000 - nagents(model)

    peak_infected = 0.0

    replicates = 5
    peak_vals = Float64[]
    dead_vals = Int[]

    for rep = 1:replicates
        model = initialize_sir(;
            Ns = [1000, 1000, 1000],
            β_und = fill(x[1], 3),
            β_det = fill(x[1]/10, 3),
            infection_period = 14,
            detection_time = round(Int, x[2]),
            death_rate = x[3],
            reinfection_probability = 0.1,
            Is = [0, 0, 1],
            seed = 42 + rep,
            n_steps = 100,
        )

        for step = 1:100
            Agents.step!(model, 1)
            frac = infected_frac(model)
            if frac > peak_infected
                peak_infected = frac
            end
        end

        push!(peak_vals, peak_infected)
        push!(dead_vals, dead_count(model))
    end

    return (mean(peak_vals), mean(dead_vals) / 3000)  #
end

cost_multi (generic function with 1 method)

## Запуск оптимизации

Используем алгоритм Borg MOEA (многокритериальный эволюционный алгоритм)
с поиском в трёхмерном пространстве параметров:
- β_und в диапазоне [0.1, 1.0]
- detection_time в диапазоне [3.0, 14.0] дней
- death_rate в диапазоне [0.01, 0.1]

Оптимизация выполняется в течение 2 минут

In [3]:
result = bboptimize(
    cost_multi,
    Method = :borg_moea,
    FitnessScheme = ParetoFitnessScheme{2}(is_minimizing = true),
    SearchRange = [
        (0.1, 1.0),    # β_und
        (3.0, 14.0),   # detection_time
        (0.01, 0.1),   # death_rate
    ],
    NumDimensions = 3,
    MaxTime = 120,  # 2 минуты
    TraceMode = :compact,
)

Starting optimization with optimizer BlackBoxOptim.BorgMOEA{BlackBoxOptim.EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}, BlackBoxOptim.ProblemEvaluator{Tuple{Float64, Float64}, BlackBoxOptim.IndexedTupleFitness{2, Float64}, BlackBoxOptim.EpsBoxArchive{2, Float64, BlackBoxOptim.EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}}, BlackBoxOptim.FunctionBasedProblem{typeof(Main.var"##278".cost_multi), BlackBoxOptim.ParetoFitnessScheme{2, Float64, true, typeof(sum)}, BlackBoxOptim.ContinuousRectSearchSpace, Nothing}}, BlackBoxOptim.FitPopulation{BlackBoxOptim.IndexedTupleFitness{2, Float64}}, BlackBoxOptim.FixedGeneticOperatorsMixture, BlackBoxOptim.RandomBound{BlackBoxOptim.ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps
pop.size=50 arch.size=0 n.restarts=0

Optimization stopped after 1 steps and 472.29 seconds
Termination reason: Max time (120.0 s) reached
Steps per second = 0.00
Function evals per second = 0.11
Improvements/step = NaN
Total function evalua

BlackBoxOptim.OptimizationResults("borg_moea", "Max time (120.0 s) reached", 1, 1.775373890381147e9, 472.2894380092621, BlackBoxOptim.ParamsDictChain[BlackBoxOptim.ParamsDictChain[Dict{Symbol, Any}(:NumDimensions => 3, :MaxTime => 120.0, :SearchRange => [(0.1, 1.0), (3.0, 14.0), (0.01, 0.1)], :TraceMode => :compact, :Method => :borg_moea, :FitnessScheme => BlackBoxOptim.ParetoFitnessScheme{2, Float64, true, typeof(sum)}(sum), :MaxFuncEvals => 0, :MaxSteps => 0),Dict{Symbol, Any}()],Dict{Symbol, Any}(:CallbackInterval => -1.0, :TargetFitness => nothing, :TraceMode => :compact, :FitnessScheme => BlackBoxOptim.ScalarFitnessScheme{true}(), :MinDeltaFitnessTolerance => 1.0e-50, :NumDimensions => :NotSpecified, :FitnessTolerance => 1.0e-8, :TraceInterval => 0.5, :MaxStepsWithoutProgress => 10000, :MaxSteps => 10000…)], 51, BlackBoxOptim.EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}([0.1, 0.1], sum), BlackBoxOptim.EpsBoxArchiveOutput{2, Float64, BlackBoxOptim.EpsBoxDominanceFitn

## Извлечение и вывод результатов

In [4]:
best = best_candidate(result)
fitness = best_fitness(result)

println("Оптимальные параметры:")
println("β_und = $(best[1])")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(best[3])")
println("Достигнутые показатели:")
println("Пик заболеваемости: $(fitness[1])")
println("Доля умерших: $(fitness[2])")

Оптимальные параметры:
β_und = 0.1427705862771556
Время выявления = 3 дней
Смертность = 0.08286843830775631
Достигнутые показатели:
Пик заболеваемости: 0.0006666666666666666
Доля умерших: 0.0


## Сохранение результатов

In [5]:
save(datadir("optimization_result.jld2"), Dict("best" => best, "fitness" => fitness))